# Apache Iceberg

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sai-guntupalli/data-engineering-interview-questions/blob/master/notebooks/topics/iceberg.ipynb)

*How to use this notebook.* Read the explanation cells and run the code cells below them. Python examples list their imports, so install what a section needs before running it. SQL and shell snippets are illustrative: they show the syntax for the platform named in that section and need that platform (or a compatible engine) to run.

## What is Apache Iceberg?

Apache Iceberg is an open table format for large analytical datasets. It is not a compute engine by itself. PySpark, Spark SQL, Flink, Trino, Snowflake, and other engines use Iceberg metadata to read and write the same table consistently.

In a PySpark data platform, the usual shape is:

```text
PySpark job -> Iceberg catalog -> Iceberg table metadata -> Parquet files in S3/GCS/ADLS/HDFS
```

Iceberg manages table metadata, snapshots, manifests, schema evolution, partition evolution, row-level deletes, and transaction commits. The data files are commonly Parquet.

Real-world example: a Databricks PySpark job writes `main.analytics.orders` every hour, Trino or another Iceberg REST client serves ad-hoc SQL, and Snowflake reads the same Unity Catalog-managed Iceberg table for BI. Iceberg is the shared table layer that makes that possible.

Under the hood an Iceberg table is four layers. The catalog holds one pointer per table: the location of the current metadata JSON file. That JSON file stores the schema, partition specs, sort orders, table properties, the snapshot log, and `current-snapshot-id`. Each snapshot points at a manifest list in Avro. The manifest list points at manifest files. Each manifest lists data files and delete files with their partition tuple, record count, file size, and column bounds.

Worked example. An append writes three Parquet files, one manifest, one manifest list, and a new metadata JSON, then swaps the catalog pointer. Snapshot `836...` becomes current and the previous snapshot id becomes its parent. A reader that loaded the old metadata JSON keeps seeing the old files until it refreshes. A query filtered on `order_ts` reads the manifest list, skips manifests whose partition summaries do not overlap, and opens only the matching Parquet files.

Common mistake: treating the catalog as the table. The catalog only stores a pointer; the table state is the metadata JSON plus the manifests and data files. If you recreate the catalog entry without the original metadata location, or delete the metadata directory, the table is lost even though the Parquet files are still on storage.

## How do you enable Iceberg in PySpark?

In Databricks with Unity Catalog enabled, you normally do not manually configure a Spark catalog for day-to-day PySpark notebooks/jobs. You use Unity Catalog's three-level namespace:

```text
catalog.schema.table
```

Example:

In [ ]:
# ILLUSTRATIVE - needs the live engine session and data this section names; read on for the pattern.
spark.sql("create schema if not exists main.analytics")

orders = spark.table("main.analytics.orders")

If your Databricks workspace supports Unity Catalog-managed Iceberg tables, create Iceberg tables directly in Unity Catalog:

In [ ]:
# ILLUSTRATIVE - needs the live engine session and data this section names; read on for the pattern.
spark.sql("""
create table if not exists main.analytics.orders (
  order_id bigint,
  customer_id bigint,
  order_ts timestamp,
  status string,
  amount decimal(12, 2)
) using iceberg
""")

Outside Databricks, or when connecting external Spark to Unity Catalog, you configure the Iceberg REST catalog endpoint for Unity Catalog. That is an interoperability setup; inside Databricks, prefer native Unity Catalog table names like `main.analytics.orders`.

Enabling Iceberg has two parts. The SQL extensions add Iceberg DDL and procedures, and a named catalog tells Spark where the pointer store lives. Set `spark.sql.extensions` to `org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions`, then register a catalog with `spark.sql.catalog.<name>`. The `<name>.type` value picks the implementation: `hive` for a Hive metastore, `hadoop` for a filesystem directory, `rest` for a catalog server, and `glue` for the AWS Glue Data Catalog.

```sql
-- Create a catalog-qualified Iceberg table. The catalog name is part of the table name.
CREATE TABLE IF NOT EXISTS prod.analytics.orders (
    order_id    BIGINT,
    customer_id BIGINT,
    order_ts    TIMESTAMP,
    status      STRING,
    amount      DECIMAL(12, 2)
)
USING iceberg
PARTITIONED BY (days(order_ts))
TBLPROPERTIES ('write.target-file-size-bytes' = '536870912');
```

In [ ]:
# ILLUSTRATIVE - requires pyspark: install to run, or read on.
from pyspark.sql import SparkSession

# REST catalog: the catalog server holds the pointer and enforces commits.
spark = (
    SparkSession.builder
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions")
    .config("spark.sql.catalog.prod", "org.apache.iceberg.spark.SparkCatalog")
    .config("spark.sql.catalog.prod.type", "rest")
    .config("spark.sql.catalog.prod.uri", "https://catalog.example.com")
    .config("spark.sql.catalog.prod.warehouse", "s3://company-lake/warehouse")
    .getOrCreate()
)

# Hive catalog: the Hive metastore stores the metadata pointer.
spark.conf.set("spark.sql.catalog.hive_prod", "org.apache.iceberg.spark.SparkCatalog")
spark.conf.set("spark.sql.catalog.hive_prod.type", "hive")
spark.conf.set("spark.sql.catalog.hive_prod.uri", "thrift://hive-metastore:9083")

# Glue catalog: AWS Glue stores databases and the metadata pointer.
spark.conf.set("spark.sql.catalog.glue_prod", "org.apache.iceberg.spark.SparkCatalog")
spark.conf.set("spark.sql.catalog.glue_prod.type", "glue")
spark.conf.set("spark.sql.catalog.glue_prod.warehouse", "s3://company-lake/warehouse")

Common mistake: creating a table in one catalog and reading it from another. A table registered under `prod` is not visible as an unqualified name in the session catalog, and the two pointers diverge if both are used for writes. Pick one catalog per table and always use the qualified name.

## How do you create and write to an Iceberg table in PySpark?

Use `spark.sql` for table DDL and PySpark DataFrameWriterV2 (`writeTo`) for table-aware writes.

Create a schema and table:

In [ ]:
# ILLUSTRATIVE - needs the live engine session and data this section names; read on for the pattern.
spark.sql("create schema if not exists main.analytics")

spark.sql("""
create table if not exists main.analytics.orders (
  order_id bigint,
  customer_id bigint,
  order_ts timestamp,
  status string,
  amount decimal(12, 2)
) using iceberg
partitioned by (days(order_ts))
""")

Append a DataFrame:

In [ ]:
# ILLUSTRATIVE - requires pyspark: install to run, or read on.
from pyspark.sql import functions as F

orders_df = (
    spark.read.json("s3://company-raw/orders/ds=2026-10-04/")
    .select(
        F.col("order_id").cast("bigint"),
        F.col("customer_id").cast("bigint"),
        F.col("order_ts").cast("timestamp"),
        F.col("status").cast("string"),
        F.col("amount").cast("decimal(12,2)"),
    )
)

orders_df.writeTo("main.analytics.orders").append()

Overwrite a single business day safely:

In [ ]:
# ILLUSTRATIVE - needs the live engine session and data this section names; read on for the pattern.
(
    orders_df
    .writeTo("main.analytics.orders")
    .overwrite(
        "order_ts >= timestamp '2026-10-04 00:00:00' "
        "and order_ts < timestamp '2026-10-05 00:00:00'"
    )
)

Interview point: `writeTo` is preferred over path-based writes because it writes through the Iceberg table abstraction and participates in Iceberg commits.

A write plans first. Spark resolves the base snapshot, computes partition values from the transform, writes data files, records column stats into manifests, then commits by creating a new metadata JSON and swapping the catalog pointer. No reader sees the new files until the swap succeeds. DataFrameWriterV2 (`writeTo`) follows this path; a path-based `write.parquet` does not, because it never writes Iceberg metadata.

```sql
CREATE TABLE IF NOT EXISTS prod.analytics.orders (
    order_id    BIGINT,
    customer_id BIGINT,
    order_ts    TIMESTAMP,
    status      STRING,
    amount      DECIMAL(12, 2)
)
USING iceberg
PARTITIONED BY (days(order_ts))
TBLPROPERTIES (
    'write.format.default' = 'parquet',
    'write.target-file-size-bytes' = '536870912'
);

INSERT INTO prod.analytics.orders
SELECT order_id, customer_id, order_ts, status, amount
FROM staging.orders_raw;
```

Common mistake: mixing path writes with Iceberg writes. `df.write.parquet('s3://.../orders')` writes files into the table location but never commits them, so the rows are invisible and later show up as orphan files. Always write through the table name.

## How does Iceberg improve data lake table management?

Plain object storage has files, but not table semantics. Iceberg adds metadata so PySpark can plan queries from table metadata instead of recursively listing directories and guessing table state.

Inspect metadata from PySpark:

In [ ]:
# ILLUSTRATIVE - needs the live engine session and data this section names; read on for the pattern.
spark.table("main.analytics.orders.snapshots").show(truncate=False)
spark.table("main.analytics.orders.files").select(
    "file_path", "record_count", "file_size_in_bytes"
).show(truncate=False)
spark.table("main.analytics.orders.history").show(truncate=False)

Real-world example: if a PySpark write job fails halfway through, readers still see the last committed snapshot. The failed job may leave orphan files, but those files are not visible as table data until a successful Iceberg commit references them.

Iceberg replaces directory listing with metadata planning. The engine reads one metadata JSON and one manifest list, then only the manifests and files whose bounds overlap the query. There is no recursive `LIST` over the table directory, so planning cost depends on manifest count, not on object-store listing speed.

Worked example: a table with 10,000 Parquet files and 20 manifests. A point query opens the metadata JSON, the manifest list, the 2 manifests whose partition ranges overlap, and 1 data file. The same data as a plain Parquet dataset would list 10,000 objects. The `files` and `manifests` metadata tables expose exactly what planning saw.

Common mistake: assuming metadata removes the need for maintenance. Manifests grow with every commit, so a table with frequent small appends accumulates thousands of manifest entries and planning slows down until `rewrite_manifests` and compaction run.

## What problems does Iceberg solve compared to plain Parquet datasets?

A plain Parquet dataset usually relies on folder layout and conventions. This causes problems with partial writes, concurrent jobs, schema changes, partition changes, deletes, updates, and slow file listing.

Path-based Parquet read:

In [ ]:
# ILLUSTRATIVE - needs the live engine session and data this section names; read on for the pattern.
plain_orders = spark.read.parquet("s3://company-lake/orders/")

Iceberg table read:

In [ ]:
# ILLUSTRATIVE - needs the live engine session and data this section names; read on for the pattern.
iceberg_orders = spark.table("main.analytics.orders")

With Iceberg, PySpark asks the catalog and Iceberg metadata which files belong to the current snapshot. With plain Parquet, Spark mostly sees files and folders. That difference matters when multiple jobs update the same dataset.

The concrete wins are atomic commits, schema evolution by field id, partition evolution, row-level deletes, and pruning from column stats. A plain Parquet dataset has none of these: a failed write leaves partial files that readers may see, a rename breaks readers, and changing the partition layout means rewriting every file.

Worked example: an append job fails after writing 40 of 100 files. In a Parquet dataset those 40 files are already visible under the partition prefix, so a reader can see half the batch. In Iceberg nothing is committed, the reader still sees the previous snapshot, and the 40 files become orphans that `remove_orphan_files` can clean up later.

Common mistake: copying or moving Parquet files underneath a live Iceberg table directory. Those files are not in any manifest, so they are invisible to readers and are treated as orphans. Add files through `add_files` or a write, never by hand.

## What is hidden partitioning in Iceberg and how do you enable it in PySpark?

Hidden partitioning means users filter on logical columns, while Iceberg stores partition transforms in table metadata. You enable it with partition transforms in table DDL.

Create a table partitioned by day of `event_ts`:

In [ ]:
# ILLUSTRATIVE - needs the live engine session and data this section names; read on for the pattern.
spark.sql("""
create table if not exists main.analytics.events (
  event_id string,
  user_id string,
  event_ts timestamp,
  event_type string
) using iceberg
partitioned by (days(event_ts))
""")

Query using the original timestamp column:

In [ ]:
# ILLUSTRATIVE - needs the live engine session and data this section names; read on for the pattern.
daily_counts = (
    spark.table("main.analytics.events")
    .where(
        (F.col("event_ts") >= F.lit("2026-10-04 00:00:00").cast("timestamp"))
        & (F.col("event_ts") < F.lit("2026-10-05 00:00:00").cast("timestamp"))
    )
    .groupBy("event_type")
    .count()
)

You do not need to create or filter on a separate `event_date` folder column. Iceberg maps the timestamp filter to the `days(event_ts)` transform and prunes files.

Other useful transforms:

In [ ]:
# ILLUSTRATIVE - needs the live engine session and data this section names; read on for the pattern.
spark.sql("""
create table if not exists main.analytics.customer_events (
  event_id string,
  customer_id string,
  region string,
  event_ts timestamp
) using iceberg
partitioned by (
  days(event_ts),
  bucket(32, customer_id),
  truncate(8, region)
)
""")

Real-world choice:

+ `days(order_ts)`: common daily reporting and backfills
+ `bucket(32, customer_id)`: frequent customer-keyed deletes, joins, or merges
+ `truncate(8, region)`: prefix-style grouping for string fields with many values

A partition transform turns a source column value into a partition value: `days(ts)` produces a date, `bucket(N, id)` produces a hash bucket from `N` and the column, and `truncate(W, s)` produces a prefix of width `W`. The transform lives in the partition spec in table metadata, so queries keep filtering the original column. Iceberg projects the filter through the transform to skip files.

Transform choices in practice:
+ `years`, `months`, `days`, `hours`: time truncation, good for range filters.
+ `bucket(N, col)`: stable hash into N buckets, good for high-cardinality equality and keyed deletes.
+ `truncate(W, col)`: prefix on strings or width-W bins on numbers.
+ `identity(col)`: partition on the column value itself.
+ `void(col)`: drop a field from the spec without removing its partition field id.

Common mistake: adding a physical `event_date` column and partitioning on it. That exposes the partition column, forces every writer to populate it, and makes the layout harder to evolve. Partition on `days(event_ts)` and keep filtering `event_ts`.

## What is partition evolution and how do you change partitions in PySpark?

Partition evolution lets you change the partition spec for new data without rewriting all old files. Historical files keep their old partition spec; new files use the new spec.

Add a bucket partition:

In [ ]:
# ILLUSTRATIVE - needs the live engine session and data this section names; read on for the pattern.
spark.sql("""
alter table main.analytics.orders
add partition field bucket(32, customer_id)
""")

Drop a partition field for future writes:

In [ ]:
# ILLUSTRATIVE - needs the live engine session and data this section names; read on for the pattern.
spark.sql("""
alter table main.analytics.orders
drop partition field days(order_ts)
""")

Then continue writing normally:

In [ ]:
# ILLUSTRATIVE - needs the live engine session and data this section names; read on for the pattern.
new_orders_df.writeTo("main.analytics.orders").append()

Important: altering the partition spec does not rewrite old data. If you want old files to be physically reorganized with the new spec from an external Spark Iceberg client, run a rewrite/compaction job. The example below assumes your external Spark catalog alias is `uc` and its warehouse is the Unity Catalog catalog `main`.

In [ ]:
# ILLUSTRATIVE - needs the live engine session and data this section names; read on for the pattern.
spark.sql("""
call uc.system.rewrite_data_files(
  table => 'analytics.orders'
)
""")

Real-world workflow: identify slow queries, change the partition spec, monitor new writes, then rewrite older data only if the performance gain is worth the cost.

Each partition spec has an id. Changing the spec creates a new spec id; new writes produce manifests for the new spec while old manifests keep the old spec. Planning reads every spec, so queries span both layouts. Old data files are not rewritten by the `ALTER`.

Common mistake: expecting `ALTER TABLE ... ADD PARTITION FIELD` to reorganize existing data. It only affects future writes. To align old files with the new layout, run a rewrite with the target spec: `CALL prod.system.rewrite_data_files(table => 'analytics.orders', options => map('output-spec-id', '1'))`.

## How does schema evolution work and how do you apply it in PySpark?

Iceberg tracks columns by stable field IDs, not only names. This makes add, drop, rename, and compatible type changes safer than unmanaged Parquet.

Add a column:

In [ ]:
# ILLUSTRATIVE - needs the live engine session and data this section names; read on for the pattern.
spark.sql("alter table main.analytics.orders add column currency string")

Rename a column:

In [ ]:
# ILLUSTRATIVE - needs the live engine session and data this section names; read on for the pattern.
spark.sql("alter table main.analytics.orders rename column amount to gross_amount")

Change a compatible type:

In [ ]:
# ILLUSTRATIVE - needs the live engine session and data this section names; read on for the pattern.
spark.sql("""
alter table main.analytics.orders
alter column gross_amount type decimal(18, 2)
""")

Drop a column:

In [ ]:
# ILLUSTRATIVE - needs the live engine session and data this section names; read on for the pattern.
spark.sql("alter table main.analytics.orders drop column currency")

Read after schema evolution:

In [ ]:
# ILLUSTRATIVE - needs the live engine session and data this section names; read on for the pattern.
orders = spark.table("main.analytics.orders")
orders.printSchema()

Interview guidance: additive columns are usually safest. Renames are safer in Iceberg than path-based Parquet, but downstream BI tools, dbt models, and notebooks still need to handle the new name.

Every column gets a unique field id in the schema. A rename changes the name mapped to an id and is metadata-only; reads project by id, and Parquet footers store ids, so old files resolve correctly after a rename. Type changes are allowed only as safe promotions such as `int` to `long`, `float` to `double`, or a wider `decimal` precision.

Common mistake: dropping a column and re-adding it with the same name. Iceberg assigns a new field id, so old files have no value for that id and readers return null for the whole history of that field. Prefer rename when you only want to change the name.

## What is a snapshot in Iceberg and how do you inspect it in PySpark?

A snapshot is a committed table version. Every successful append, overwrite, merge, delete, or schema change creates a new table state.

Inspect snapshots:

In [ ]:
# ILLUSTRATIVE - needs the live engine session and data this section names; read on for the pattern.
snapshots = spark.table("main.analytics.orders.snapshots")
snapshots.select("committed_at", "snapshot_id", "operation", "summary").show(truncate=False)

Inspect history:

In [ ]:
# ILLUSTRATIVE - needs the live engine session and data this section names; read on for the pattern.
spark.table("main.analytics.orders.history").orderBy(F.desc("made_current_at")).show(truncate=False)

Inspect files in the current table:

In [ ]:
# ILLUSTRATIVE - needs the live engine session and data this section names; read on for the pattern.
spark.table("main.analytics.orders.files").select(
    "content", "file_path", "record_count", "file_size_in_bytes"
).show(truncate=False)

Real-world use: after a bad PySpark deployment, find the snapshot from before the deployment and compare row counts or metrics.

A snapshot records a manifest list, a parent snapshot id, a sequence number, a timestamp, and a summary map such as `added-data-files`, `added-records`, `total-records`, and `total-data-files`. Snapshot ids are 64-bit longs assigned at commit time, usually derived from the commit timestamp and made monotonic. The `main` branch points at exactly one current snapshot.

Common mistake: thinking a snapshot copies the data. A snapshot references immutable files, so only new files consume storage. That is also why expired and orphan files must be deleted by maintenance rather than by dropping a snapshot.

## What is time travel and how do you use it in PySpark?

Time travel queries a table as it existed at a previous snapshot or timestamp. Use it for debugging, audits, regression checks, and reproducing reports.

Read by snapshot ID:

In [ ]:
# ILLUSTRATIVE - needs the live engine session and data this section names; read on for the pattern.
old_orders = (
    spark.read
    .option("snapshot-id", "1234567890123456789")
    .format("iceberg")
    .load("main.analytics.orders")
)

Read by timestamp:

In [ ]:
# ILLUSTRATIVE - needs the live engine session and data this section names; read on for the pattern.
orders_as_of = (
    spark.read
    .option("as-of-timestamp", "1791098400000")
    .format("iceberg")
    .load("main.analytics.orders")
)

Compare old and current statuses:

In [ ]:
# ILLUSTRATIVE - needs the live engine session and data this section names; read on for the pattern.
current_orders = spark.table("main.analytics.orders").select("order_id", "status")
old_orders = old_orders.select(
    "order_id",
    F.col("status").alias("old_status"),
)

changed = (
    current_orders.alias("c")
    .join(old_orders.alias("o"), "order_id")
    .where(F.col("c.status") != F.col("o.old_status"))
)

Operational note: time travel only works while old snapshots and files are retained. If maintenance expires old snapshots and removes old files, very old versions are no longer queryable.

Time travel resolves a version to a snapshot: by id, by timestamp (the latest snapshot committed at or before that time on an ancestor path), or by branch or tag name. Iceberg then plans the scan against that snapshot's manifests. In Spark SQL use `VERSION AS OF` and `TIMESTAMP AS OF`; in the DataFrame reader use `snapshot-id`, `as-of-timestamp`, `branch`, or `tag`.

```sql
-- Snapshot id from the snapshots metadata table.
SELECT * FROM prod.analytics.orders VERSION AS OF 8364525912309844123;

-- A wall-clock time: the latest snapshot at or before this timestamp.
SELECT * FROM prod.analytics.orders TIMESTAMP AS OF '2026-10-04 12:00:00';

-- A named branch or tag.
SELECT * FROM prod.analytics.orders VERSION AS OF 'finance_close_2026_09';
```

Common mistake: assuming any past timestamp is queryable. Time travel only reaches snapshots that still exist, and `expire_snapshots` removes the rest. A query for an expired snapshot fails; it does not silently fall back to current data.

## How does Iceberg handle concurrent writes?

Iceberg uses optimistic concurrency control. A writer plans changes from a base snapshot, writes new files, then tries to commit new metadata. If another writer committed a conflicting change first, the commit fails or retries instead of silently corrupting the table.

Example scenario:

```text
PySpark job A reads snapshot 10 and prepares append
PySpark job B reads snapshot 10 and commits snapshot 11
PySpark job A attempts commit; Iceberg checks whether A can safely commit on top of snapshot 11
```

Practical PySpark guidance:

+ keep batch jobs partition-scoped where possible
+ avoid two jobs rewriting the same table partition at the same time
+ make merge jobs idempotent
+ retry commit conflicts at the orchestration layer when safe
+ compact after heavy updates/deletes

For append-only workloads, retries are usually easier. For overlapping `merge`, `delete`, and overwrite workloads, conflicts are more likely.

Commits are optimistic. A writer reads the current metadata, creates new files and a candidate metadata JSON, then asks the catalog for an atomic pointer swap. If the pointer moved, the catalog rejects the commit and the writer validates whether the concurrent change is compatible. Append-style commits usually rebase and retry; conflicting isolation-level changes fail. Retries are governed by `commit.retry.num-retries`, which defaults to 4.

Common mistake: treating a commit conflict as data corruption and rerunning the job blindly. Iceberg retries internally; if a job still fails, re-read the current snapshot and make the merge idempotent instead of overwriting files by path.

## What are equality deletes and positional deletes, and how do you use deletes in PySpark?

Iceberg supports row-level deletes by writing delete files instead of immediately rewriting every affected data file.

+ Equality deletes mark rows as deleted based on column values, such as `order_id = 1001`.
+ Positional deletes mark specific row positions in specific data files.

Most PySpark users do not manually create delete files. They use SQL commands or `MERGE INTO`, and the engine writes the appropriate Iceberg files.

Delete old cancelled orders:

In [ ]:
# ILLUSTRATIVE - needs the live engine session and data this section names; read on for the pattern.
spark.sql("""
delete from main.analytics.orders
where status = 'cancelled'
  and order_ts < timestamp '2026-01-01 00:00:00'
""")

Delete a customer for a privacy request:

In [ ]:
# ILLUSTRATIVE - needs the live engine session and data this section names; read on for the pattern.
customer_id = "C123"
spark.sql(f"""
delete from main.analytics.customer_events
where customer_id = '{customer_id}'
""")

Configure merge-on-read delete/update behavior:

In [ ]:
# ILLUSTRATIVE - needs the live engine session and data this section names; read on for the pattern.
spark.sql("""
alter table main.analytics.orders
set tblproperties (
  'write.delete.mode' = 'merge-on-read',
  'write.update.mode' = 'merge-on-read',
  'write.merge.mode' = 'merge-on-read'
)
""")

Operational note: merge-on-read can make writes faster, but many delete files can slow reads. Follow heavy delete/update workloads with compaction.

In v2 tables Iceberg adds sequence numbers. A delete file applies only to data files with a lower sequence number, so a later insert of the same key survives. Positional deletes store `(file_path, row_position)` pairs and are usually produced by `MERGE`; equality deletes store the deleted key values and are common with Flink. With `write.merge.mode=merge-on-read` the engine writes delete files instead of rewriting data files, and readers merge them at scan time.

Common mistake: leaving a merge-on-read table without compaction. Every merge adds delete files, readers must apply more of them, and a data file with many applied deletes becomes expensive. Run `rewrite_data_files` to materialize the deletes and drop the delete files.

## How do upserts and merges work with Iceberg in PySpark?

Use `MERGE INTO` from PySpark through `spark.sql`. Deduplicate CDC input first so each target key has one winning source record.

Deduplicate CDC input:

In [ ]:
# ILLUSTRATIVE - requires pyspark: install to run, or read on.
from pyspark.sql.window import Window

cdc = spark.table("main.bronze.orders_cdc")

window = Window.partitionBy("order_id").orderBy(
    F.col("source_lsn").desc(),
    F.col("ingested_at").desc(),
)

deduped = (
    cdc
    .withColumn("rn", F.row_number().over(window))
    .where(F.col("rn") == 1)
    .drop("rn")
)

deduped.createOrReplaceTempView("orders_cdc_deduped")

Merge into the Iceberg table:

In [ ]:
# ILLUSTRATIVE - needs the live engine session and data this section names; read on for the pattern.
spark.sql("""
merge into main.analytics.orders as target
using orders_cdc_deduped as source
on target.order_id = source.order_id
when matched and source.op = 'D' then delete
when matched then update set
  customer_id = source.customer_id,
  order_ts = source.order_ts,
  status = source.status,
  amount = source.amount
when not matched and source.op <> 'D' then insert (
  order_id, customer_id, order_ts, status, amount
) values (
  source.order_id, source.customer_id, source.order_ts, source.status, source.amount
)
""")

Real-world guidance:

+ merge on stable business keys
+ dedupe CDC input before merge
+ include delete/tombstone logic
+ compact after large merge jobs
+ monitor both data files and delete files

A merge is a join on the merge key. Iceberg plans the target files that can contain matching keys, shuffles source and target by key, applies the matched and not-matched actions, and commits one snapshot. Partition pruning on the merge key limits how many target files are rewritten. `write.merge.mode` selects copy-on-write (rewrite touched files) or merge-on-read (write delete files plus inserts).

Common mistake: merging a CDC stream that has several rows per key. The merge sees duplicates and the outcome depends on join order. Deduplicate to one winning row per key first, as the example does with `row_number()`.

## Why does the small files problem happen and how do you mitigate it in PySpark?

Small files happen when many jobs write small batches, streaming jobs commit frequently, partitioning is too granular, or Spark uses too many writer tasks.

Set target file size:

In [ ]:
# ILLUSTRATIVE - needs the live engine session and data this section names; read on for the pattern.
spark.sql("""
alter table main.analytics.orders
set tblproperties (
  'write.target-file-size-bytes' = '536870912'
)
""")

Control write parallelism before append:

In [ ]:
# ILLUSTRATIVE - needs the live engine session and data this section names; read on for the pattern.
(
    orders_df
    .repartition(32, F.to_date("order_ts"))
    .writeTo("main.analytics.orders")
    .append()
)

Compact small files:

In [ ]:
# ILLUSTRATIVE - needs the live engine session and data this section names; read on for the pattern.
spark.sql("""
call uc.system.rewrite_data_files(
  table => 'analytics.orders',
  options => map('target-file-size-bytes', '536870912')
)
""")

Interview answer: prevention is better than cleanup. Choose sane partitioning, batch enough data per commit, and tune Spark writer parallelism.

Every commit adds at least one manifest and one data file per writing task. Small files hurt twice: more manifests slow planning, and more files raise per-file open cost. The default `write.target-file-size-bytes` is 512 MB and `read.split.open-file-cost` is 4 MB, so a table of 1 KB files pays an open cost far larger than the data it reads.

In [ ]:
# A file-size histogram is the quickest read on small-file growth.
files = [
    {"path": "data/00000.parquet", "size_mb": 1.2},
    {"path": "data/00001.parquet", "size_mb": 3.4},
    {"path": "data/00002.parquet", "size_mb": 480.0},
    {"path": "data/00003.parquet", "size_mb": 512.0},
]

tiny = [f for f in files if f["size_mb"] < 128]
total_mb = sum(f["size_mb"] for f in files)
print(f"{len(tiny)} of {len(files)} files are under 128 MB")
print(f"table size = {total_mb:.1f} MB")
assert len(tiny) == 2 and round(total_mb, 1) == 996.6

Common mistake: setting a target file size and assuming existing files change. The property only guides new writes; existing small files need `rewrite_data_files`, which bin-packs file groups down to the target size.

## What maintenance operations are common for Iceberg tables in PySpark?

Iceberg tables need periodic maintenance after streaming writes, merges, deletes, and frequent schema/partition changes.

The examples below assume an external Spark Iceberg REST catalog alias named `uc` pointed at the Unity Catalog catalog `main`. If you are running directly inside Databricks, use the platform-supported maintenance commands and policies for your table type/runtime.

Compact data files:

In [ ]:
# ILLUSTRATIVE - needs the live engine session and data this section names; read on for the pattern.
spark.sql("""
call uc.system.rewrite_data_files(
  table => 'analytics.orders'
)
""")

Rewrite manifests:

In [ ]:
# ILLUSTRATIVE - needs the live engine session and data this section names; read on for the pattern.
spark.sql("""
call uc.system.rewrite_manifests(
  table => 'analytics.orders'
)
""")

Expire old snapshots:

In [ ]:
# ILLUSTRATIVE - needs the live engine session and data this section names; read on for the pattern.
spark.sql("""
call uc.system.expire_snapshots(
  table => 'analytics.orders',
  older_than => timestamp '2026-09-01 00:00:00',
  retain_last => 10
)
""")

Remove orphan files:

In [ ]:
# ILLUSTRATIVE - needs the live engine session and data this section names; read on for the pattern.
spark.sql("""
call uc.system.remove_orphan_files(
  table => 'analytics.orders',
  older_than => timestamp '2026-09-01 00:00:00'
)
""")

Real-world schedule:

+ compact high-write tables daily or hourly depending on volume
+ expire snapshots after the required rollback/audit period
+ remove orphan files carefully after confirming no active jobs need them
+ monitor metadata size, file counts, and planning latency

Each procedure has a narrow job. `rewrite_data_files` bin-packs small and delete-heavy files into target-size files. `rewrite_manifests` re-clusters manifest entries by partition to speed planning. `expire_snapshots` deletes snapshots and the files only they reference, with defaults of `older_than` five days ago and `retain_last` 1. `remove_orphan_files` deletes files no snapshot references, defaulting to older than three days ago.

```sql
-- Compact: bin-pack small files and materialize merge-on-read deletes.
CALL prod.system.rewrite_data_files(
  table => 'analytics.orders',
  options => map('target-file-size-bytes', '536870912', 'min-input-files', '5')
);

-- Re-cluster manifests so planning can skip more of them.
CALL prod.system.rewrite_manifests(table => 'analytics.orders', sort_by => array('order_ts_day'));

-- Retire snapshots past the audit window but always keep the last 10.
CALL prod.system.expire_snapshots(
  table => 'analytics.orders',
  older_than => TIMESTAMP '2026-09-01 00:00:00',
  retain_last => 10
);

-- Inspect orphan candidates first.
CALL prod.system.remove_orphan_files(
  table => 'analytics.orders',
  older_than => TIMESTAMP '2026-09-01 00:00:00',
  dry_run => true
);
```

Common mistake: running `remove_orphan_files` with a cutoff inside the window of an in-flight job. A job that has written files but not yet committed will see its files deleted. Set `older_than` beyond the longest job and dry-run first.

## What is an Iceberg catalog and how do you configure one for PySpark?

An Iceberg catalog stores namespaces and the pointer to each table's current metadata file. It is how PySpark finds and commits to tables.

In Databricks, Unity Catalog is the default recommended catalog for governed Delta and Iceberg tables. You use three-part names:

```text
main.analytics.orders
prod_lake.finance.fact_revenue
```

Create and use a Unity Catalog schema/table from PySpark:

In [ ]:
# ILLUSTRATIVE - needs the live engine session and data this section names; read on for the pattern.
spark.sql("create schema if not exists main.analytics")

spark.sql("""
create table if not exists main.analytics.orders (
  order_id bigint,
  customer_id bigint,
  order_ts timestamp,
  amount decimal(12, 2)
) using iceberg
""")

For external Iceberg-compatible engines, Unity Catalog exposes an Iceberg REST catalog interface. A non-Databricks Spark job connecting through that REST interface would use Spark catalog configuration similar to:

In [ ]:
# ILLUSTRATIVE - needs the live engine session and data this section names; read on for the pattern.
spark = (
    SparkSession.builder
    .config("spark.sql.catalog.uc", "org.apache.iceberg.spark.SparkCatalog")
    .config("spark.sql.catalog.uc.type", "rest")
    .config("spark.sql.catalog.uc.uri", "https://<databricks-workspace>/api/2.1/unity-catalog/iceberg-rest")
    .config("spark.sql.catalog.uc.rest.auth.type", "oauth2")
    .config("spark.sql.catalog.uc.oauth2-server-uri", "https://<databricks-workspace>/oidc/v1/token")
    .config("spark.sql.catalog.uc.credential", "<oauth-client-id>:<oauth-client-secret>")
    .config("spark.sql.catalog.uc.warehouse", "main")
    .config("spark.sql.catalog.uc.scope", "all-apis")
    .getOrCreate()
)

Other catalog choices still exist, such as Hive metastore, AWS Glue, Apache Polaris, Hadoop catalog, REST catalog, and Nessie. But if your platform is Databricks-centered, use Unity Catalog first because it centralizes governance, permissions, lineage, and external Iceberg REST access.

Interview point: catalog choice affects authorization, interoperability, disaster recovery, and how many engines can safely share the same tables.

The catalog is the concurrency control point. Hive uses a metastore transaction, REST uses the catalog server's compare-and-swap, Glue uses `UpdateTable` with a version check, and Hadoop uses atomic rename plus a `version-hint.text` file. The REST protocol is what lets engines with no shared library, such as Snowflake or Trino, commit under the same rules.

```sql
CREATE NAMESPACE IF NOT EXISTS prod.analytics;

CREATE TABLE IF NOT EXISTS prod.analytics.orders (
    order_id BIGINT,
    order_ts TIMESTAMP
)
USING iceberg
PARTITIONED BY (days(order_ts));
```

Common mistake: using a Hadoop catalog on object storage for concurrent writes. S3 has no atomic rename, so two writers can publish competing metadata files. Use a REST, Hive, or Glue catalog when more than one writer commits.

## How would you migrate an existing Parquet dataset to Iceberg with PySpark?

The safest approach is usually to create a new Iceberg table and rewrite the data into it. This lets you enforce schema, clean bad records, and choose partitioning deliberately.

Read legacy Parquet:

In [ ]:
# ILLUSTRATIVE - needs the live engine session and data this section names; read on for the pattern.
legacy = spark.read.parquet("s3://company-lake/legacy/orders/")

Create the Iceberg target:

In [ ]:
# ILLUSTRATIVE - needs the live engine session and data this section names; read on for the pattern.
spark.sql("""
create table if not exists main.analytics.orders_iceberg (
  order_id bigint,
  customer_id bigint,
  order_ts timestamp,
  status string,
  amount decimal(12, 2)
) using iceberg
partitioned by (days(order_ts))
""")

Clean and write:

In [ ]:
# ILLUSTRATIVE - needs the live engine session and data this section names; read on for the pattern.
cleaned = legacy.select(
    F.col("order_id").cast("bigint"),
    F.col("customer_id").cast("bigint"),
    F.col("order_ts").cast("timestamp"),
    F.col("status").cast("string"),
    F.col("amount").cast("decimal(12,2)"),
)

cleaned.writeTo("main.analytics.orders_iceberg").append()

Validate:

In [ ]:
# ILLUSTRATIVE - needs the live engine session and data this section names; read on for the pattern.
legacy_count = legacy.count()
iceberg_count = spark.table("main.analytics.orders_iceberg").count()

legacy_amount = legacy.select(F.sum("amount")).first()[0]
iceberg_amount = spark.table("main.analytics.orders_iceberg").select(F.sum("amount")).first()[0]

Cutover strategy:

+ backfill historical data into Iceberg
+ validate row counts, sums, keys, and sample records
+ pause or dual-write source ingestion briefly
+ apply incremental changes
+ switch readers to the Iceberg table
+ keep the old dataset read-only until confidence is high

There are three paths. Rewriting through `INSERT` gives the cleanest metadata and partition layout. `add_files` registers existing files without moving them, but it does not validate the schema and Iceberg then owns those files. `migrate` replaces an existing Hive or Spark table with an Iceberg table that keeps the source data files; `snapshot` makes a lightweight test copy that shares the same files.

```sql
-- In-place migration: the source table is retained as a backup by default.
CALL prod.system.migrate('spark_catalog.analytics.orders_parquet');

-- Test first with a lightweight snapshot table that shares the same files.
CALL prod.system.snapshot(
  source_table => 'spark_catalog.analytics.orders_parquet',
  table => 'analytics.orders_snapshot'
);

-- Register existing Parquet files as Iceberg data files without moving them.
CALL prod.system.add_files(
  table => 'analytics.orders',
  source_table => '`parquet`.`s3://company-lake/legacy/orders`'
);

-- Register a metadata JSON that already exists on storage.
CALL prod.system.register_table(
  table => 'analytics.orders',
  metadata_file => 's3://company-lake/warehouse/analytics/orders/metadata/00001-abc.metadata.json'
);
```

Common mistake: migrating by pointing a new table at the old directory and continuing to write with the old path-based job. The two writers never see each other's commits. Cut writes over to the Iceberg table before retiring the Parquet writer.

## Iceberg vs Delta Lake: what is the difference with examples?

Iceberg and Delta Lake both make data lake tables more reliable than plain Parquet, but they make different ecosystem and interoperability trade-offs.

Create the same orders table as Iceberg:

In [ ]:
# ILLUSTRATIVE - needs the live engine session and data this section names; read on for the pattern.
spark.sql("""
create table if not exists main.analytics.orders_iceberg (
  order_id bigint,
  customer_id bigint,
  order_ts timestamp,
  amount decimal(12, 2)
) using iceberg
partitioned by (days(order_ts))
""")

orders_df.writeTo("main.analytics.orders_iceberg").append()

Create the same orders table as Delta:

In [ ]:
# ILLUSTRATIVE - needs the live engine session and data this section names; read on for the pattern.
(
    orders_df
    .write
    .format("delta")
    .mode("append")
    .partitionBy("order_date")
    .save("s3://company-lake/delta/orders")
)

Key practical differences:

+ Iceberg focuses on open, multi-engine table interoperability with hidden partitioning and strong partition evolution.
+ Delta Lake is deeply integrated with the Delta/Databricks ecosystem and has mature Delta-specific optimization features.
+ Iceberg partitioning is metadata-driven, so users can filter `order_ts` while Iceberg prunes `days(order_ts)`.
+ Traditional Delta partitioning often exposes physical partition columns like `order_date`, though Delta UniForm can expose Delta tables to Iceberg readers.

Example interview answer:

```text
If Spark writes the table, Trino reads it, Snowflake also needs access, and the organization wants an open table format across engines, Iceberg is a strong fit.
```


The metadata design is the difference:
+ Iceberg keeps a metadata JSON, a manifest list, and manifest files, and stores partition transforms in the spec.
+ Delta keeps a `_delta_log` of JSON commits with periodic Parquet checkpoints, and traditional Delta partitioning uses physical partition columns.
+ Iceberg is designed around the REST catalog protocol so many engines can commit; Delta is designed around the Delta protocol and the Databricks ecosystem.
+ Deletes: Iceberg writes positional or equality delete files; Delta writes deletion vectors.

Common mistake: assuming the format alone guarantees interoperability. What matters is whether the engines you use support the catalog and protocol, and whether they can write, not just read. Iceberg gives you the REST protocol; Delta gives you UniForm.

## How do you make a Delta table Iceberg-compatible?

The common modern approach is Delta UniForm, which lets a Delta table generate Iceberg-compatible metadata so Iceberg readers can read it. The Delta table remains a Delta table; Iceberg clients should treat it as read-only unless your platform explicitly supports more.

Create or alter a Delta table with UniForm enabled from PySpark:

In [ ]:
# ILLUSTRATIVE - needs the live engine session and data this section names; read on for the pattern.
spark.sql("""
create table if not exists delta.`s3://company-lake/delta/orders_uniform` (
  order_id bigint,
  customer_id bigint,
  order_ts timestamp,
  order_date date,
  amount decimal(12, 2)
) using delta
tblproperties (
  'delta.enableIcebergCompatV2' = 'true',
  'delta.universalFormat.enabledFormats' = 'iceberg'
)
partitioned by (order_date)
""")

Or enable it on an existing Delta table:

In [ ]:
# ILLUSTRATIVE - needs the live engine session and data this section names; read on for the pattern.
spark.sql("""
alter table delta.`s3://company-lake/delta/orders_uniform`
set tblproperties (
  'delta.enableIcebergCompatV2' = 'true',
  'delta.universalFormat.enabledFormats' = 'iceberg'
)
""")

Write data normally as Delta:

In [ ]:
# ILLUSTRATIVE - needs the live engine session and data this section names; read on for the pattern.
(
    orders_df
    .withColumn("order_date", F.to_date("order_ts"))
    .write
    .format("delta")
    .mode("append")
    .save("s3://company-lake/delta/orders_uniform")
)

What happens next:

+ Delta remains the source of truth.
+ UniForm asynchronously generates Iceberg-compatible metadata.
+ Iceberg engines can read the table using the generated Iceberg metadata.
+ Do not assume Iceberg writers can safely mutate the table; treat Iceberg access as read-oriented unless your vendor/runtime explicitly supports otherwise.

If you need a true native Iceberg table instead of a Delta table exposed as Iceberg, rewrite the Delta table into Iceberg:

In [ ]:
# ILLUSTRATIVE - needs the live engine session and data this section names; read on for the pattern.
delta_orders = spark.read.format("delta").load("s3://company-lake/delta/orders_uniform")

spark.sql("""
create table if not exists main.analytics.orders_iceberg (
  order_id bigint,
  customer_id bigint,
  order_ts timestamp,
  order_date date,
  amount decimal(12, 2)
) using iceberg
partitioned by (days(order_ts))
""")

delta_orders.writeTo("main.analytics.orders_iceberg").append()

UniForm generates metadata, it does not convert the table. Delta stays the source of truth and keeps writing `_delta_log`; a background process writes Iceberg metadata beside it that points at the same Parquet files. Iceberg readers see snapshots that track Delta commits. Supported input formats and table features are restricted, so check the compatibility list before enabling it.

Common mistake: writing to a UniForm table with an Iceberg writer while Delta also writes. Both formats maintain their own commit logs over the same files, and the two views diverge. Pick one writer.

## How do you access a PySpark-created Iceberg table in Snowflake?

A typical pattern is:

1. Databricks PySpark writes an Iceberg table registered in Unity Catalog.
2. A Databricks service principal is granted access to the Unity Catalog catalog/schema/table.
3. Snowflake creates an Iceberg REST catalog integration that points to the Databricks Unity Catalog Iceberg REST endpoint.
4. Snowflake creates a catalog-linked database that syncs Unity Catalog namespaces and Iceberg tables.
5. Snowflake queries the Iceberg table without copying the data.

Databricks PySpark side: create a Unity Catalog-managed Iceberg table:

In [ ]:
# ILLUSTRATIVE - needs the live engine session and data this section names; read on for the pattern.
spark.sql("create schema if not exists main.analytics")

spark.sql("""
create table if not exists main.analytics.orders (
  order_id bigint,
  customer_id bigint,
  order_ts timestamp,
  amount decimal(12, 2)
) using iceberg
partitioned by (days(order_ts))
""")

orders_df.writeTo("main.analytics.orders").append()

Databricks permissions side: grant the Snowflake service principal the relevant Unity Catalog privileges. The exact principal name depends on how your Databricks service principal is represented.

```sql
GRANT USE CATALOG ON CATALOG main TO `<application_id>`;
GRANT USE SCHEMA ON SCHEMA main.analytics TO `<application_id>`;
GRANT EXTERNAL USE SCHEMA ON SCHEMA main.analytics TO `<application_id>`;
GRANT SELECT ON TABLE main.analytics.orders TO `<application_id>`;
```

Snowflake side: create a catalog integration that connects to the Databricks Unity Catalog Iceberg REST endpoint:

```sql
CREATE OR REPLACE CATALOG INTEGRATION databricks_uc_iceberg
  CATALOG_SOURCE = ICEBERG_REST
  TABLE_FORMAT = ICEBERG
  REST_CONFIG = (
    CATALOG_URI = 'https://<databricks-workspace-url>/api/2.1/unity-catalog/iceberg-rest'
    CATALOG_NAME = 'main'
    ACCESS_DELEGATION_MODE = VENDED_CREDENTIALS
  )
  REST_AUTHENTICATION = (
    TYPE = OAUTH
    OAUTH_TOKEN_URI = 'https://<databricks-workspace-url>/oidc/v1/token'
    OAUTH_CLIENT_ID = '<databricks-service-principal-application-id>'
    OAUTH_CLIENT_SECRET = '<databricks-service-principal-secret>'
    OAUTH_ALLOWED_SCOPES = ('all-apis')
  )
  ENABLED = TRUE;
```

Then create a catalog-linked database:

```sql
CREATE DATABASE databricks_uc
  LINKED_CATALOG = (
    CATALOG = 'databricks_uc_iceberg'
    BLOCKED_NAMESPACES = ('information_schema')
  );
```

After sync, Snowflake exposes Unity Catalog schemas and Iceberg tables under the linked database. For example, `main.analytics.orders` in Unity Catalog can appear under the Snowflake catalog-linked database as an `analytics.orders` table, depending on the sync and naming configuration.

Important operational points:

+ PySpark and Snowflake must point to the same Unity Catalog Iceberg table metadata.
+ With catalog-vended credentials, Unity Catalog can provide Snowflake temporary scoped credentials to access the underlying data.
+ Keep ownership clear. If Databricks owns writes, treat Snowflake primarily as a reader unless you intentionally configure bidirectional/write access.
+ Grant only the Unity Catalog privileges Snowflake needs.
+ Run table maintenance from the owning platform so metadata, snapshots, and data files stay healthy.

Snowflake acts as an Iceberg REST client. A catalog integration names the REST endpoint and the OAuth credentials; a catalog-linked database syncs namespaces and tables. With `ACCESS_DELEGATION_MODE = VENDED_CREDENTIALS`, the catalog hands Snowflake short-lived storage credentials, so no long-lived cloud key is stored in Snowflake.

Common mistake: granting `SELECT` but not `EXTERNAL USE SCHEMA`, or the reverse, so the table appears in the linked database but queries fail. Grant `USE CATALOG`, `USE SCHEMA`, `EXTERNAL USE SCHEMA`, and `SELECT`, and run maintenance from the owning platform.

## Iceberg vs Delta vs Hudi: when would you choose Iceberg?

Choose Iceberg when you need an open, engine-neutral table format with strong metadata handling, hidden partitioning, partition evolution, schema evolution, time travel, and multi-engine access.

Example strong Iceberg fit:

```text
PySpark writes curated tables
Trino serves ad-hoc queries
Snowflake reads selected tables for BI
Flink writes streaming updates
Data lives in S3
Governance uses Databricks Unity Catalog and its Iceberg REST catalog interface
```

Choose carefully:

+ Iceberg: multi-engine lakehouse, open table standard, hidden partitioning, partition/schema evolution
+ Delta: strong fit in Delta/Databricks-centered environments, with UniForm when external Iceberg readers are needed
+ Hudi: strong fit for CDC-heavy ingestion and incremental pull patterns

Interview answer: do not say one is universally better. Tie the choice to engines, governance, update/delete needs, streaming requirements, operational tooling, and team experience.

Each format optimizes for a different core abstraction. Hudi is built around a timeline and copy-on-write or merge-on-read table types, tuned for upsert-heavy ingestion and incremental pull. Delta is built around the transaction log and tight Databricks integration. Iceberg is built around the four-layer metadata tree and the REST catalog protocol, tuned for many engines reading and writing one table.

Common mistake: picking a format from a feature checklist instead of the operational constraint. The deciding questions are which engines must write, which catalog is shared, and who will run compaction.

## How do branching and tagging work in Iceberg?

A branch is a named mutable reference to a snapshot; a tag is a named immutable reference. Both live in the `refs` map in table metadata. `main` is the default branch and never expires. Writes can target a branch for a write-audit-publish flow: write to an audit branch, validate, then fast-forward `main`.

```sql
-- Tag a snapshot for an auditable report.
ALTER TABLE prod.analytics.orders
  CREATE TAG finance_close_2026_09 AS OF VERSION 8364525912309844123 RETAIN 180 DAYS;

-- Branch for write-audit-publish.
ALTER TABLE prod.analytics.orders
  CREATE BRANCH audit_2026_10_04 RETAIN 7 DAYS WITH SNAPSHOT RETENTION 2 SNAPSHOTS;

-- Write to and read from the branch.
INSERT INTO prod.analytics.orders.branch_audit_2026_10_04
VALUES (1001, 42, TIMESTAMP '2026-10-04 09:00:00', 'new', 19.99);

SELECT * FROM prod.analytics.orders VERSION AS OF 'audit_2026_10_04';

-- Inspect references.
SELECT * FROM prod.analytics.orders.refs;
```

Common mistake: assuming a branch or tag protects its files forever. A reference keeps the snapshots it points at past the table's snapshot-age rule, but a tag created with `RETAIN N DAYS` stops protecting them after N days, and `expire_snapshots` can then remove them.

## How do you tune Iceberg tables for performance?

Tune in this order: prune more, then read fewer and larger files, then plan faster. Pruning depends on the partition spec and column stats. File efficiency depends on the size histogram. Planning depends on manifest count. Delete-file accumulation and catalog overhead come last. Every one of these is measurable from the metadata tables before you change anything.

The numbers to collect: `files` gives `file_size_in_bytes` and `content` (0 data, 1 position deletes, 2 equality deletes); `manifests` gives manifest count and per-manifest file counts; `snapshots` gives commit age, operation, and summary counts; `partitions` gives file and record counts per partition. A healthy table has files near `write.target-file-size-bytes`, few delete files, and manifest counts in the hundreds, not tens of thousands.

```sql
-- File-size histogram: how much of the table is in small files?
SELECT CASE
         WHEN file_size_in_bytes < 16 * 1024 * 1024 THEN 'tiny (<16MB)'
         WHEN file_size_in_bytes < 128 * 1024 * 1024 THEN 'small (<128MB)'
         WHEN file_size_in_bytes < 640 * 1024 * 1024 THEN 'target'
         ELSE 'large'
       END AS size_bucket,
       COUNT(*) AS files,
       ROUND(SUM(file_size_in_bytes) / POWER(1024, 3), 1) AS gib
FROM prod.analytics.orders.files
WHERE content = 0
GROUP BY 1;

-- Delete-file accumulation from merge-on-read writes.
SELECT content, COUNT(*) AS files, ROUND(AVG(record_count), 0) AS avg_rows
FROM prod.analytics.orders.files
WHERE content > 0
GROUP BY 1;

-- Planning pressure: manifests and the files they track.
SELECT COUNT(*) AS manifests,
       SUM(added_data_files_count + existing_data_files_count) AS tracked_files
FROM prod.analytics.orders.manifests;

-- Snapshot age and operation, to size the expiry window.
SELECT committed_at, snapshot_id, operation,
       summary['total-data-files'] AS total_data_files
FROM prod.analytics.orders.snapshots
ORDER BY committed_at DESC;
```

Common mistake: tuning by raising the target file size only. A larger target makes new files bigger but leaves the existing small files, so the histogram barely moves. Pair the property change with one `rewrite_data_files` run, then re-measure the same metadata table.

## How do you choose a partition transform?

Match the transform to the dominant predicate. Date-range filters want a time transform. High-cardinality equality filters and keyed merges want `bucket`. Low-cardinality category filters want `identity` or `truncate`. Aim for roughly one to a few million rows per partition: enough files to prune, few enough partitions to avoid a metadata explosion.

Hidden partitioning means the query never mentions the partition column, so the transform can change later. Prefer one time transform plus at most one `bucket`; each extra field multiplies the partition count and the manifest count. `truncate(8, region)` splits long strings into prefixes, while `bucket(32, customer_id)` spreads customers evenly and tolerates skew.

Common mistake: partitioning a high-cardinality column with `identity`. A customer-id identity partition creates millions of tiny partitions and a metadata explosion. Use `bucket(N, customer_id)` instead and pick N from the data volume.

## How do you keep manifests and metadata small?

Scan planning cost grows with manifest count, and manifests grow with every commit. `rewrite_manifests` rewrites them clustered by partition so planning can skip whole manifests. `sort_by` clusters manifests by the partition field your queries filter on most. The goal is manifests near `read.split.metadata-target-size` (32 MB), not thousands of tiny ones.

Metadata JSON files also accumulate. `write.metadata.previous-versions-max` defaults to 100 and caps how many are tracked; `write.metadata.delete-after-commit.enabled` defaults to false and deletes the oldest tracked versions after each commit. Turn both on when commit rate is high, but keep a window long enough for rollback. `read.split.planning-lookback` defaults to 10.

```sql
-- Cap metadata history and let commits delete old metadata JSON.
ALTER TABLE prod.analytics.orders SET TBLPROPERTIES (
  'write.metadata.previous-versions-max' = '50',
  'write.metadata.delete-after-commit.enabled' = 'true'
);

-- Cluster manifests by the partition field used in most filters.
CALL prod.system.rewrite_manifests(
  table => 'analytics.orders',
  sort_by => array('order_ts_day')
);
```

Common mistake: rewriting manifests continuously. Manifest rewrite is a Spark job and creates a new snapshot each time, which itself adds metadata. Run it when manifest count or planning time grows, not on every commit.

## Iceberg exercises

These exercises use only the Python standard library and model Iceberg maintenance with small, runnable calculations. Each one names the task, gives a hint, states the expected result, and prints or asserts the answer. Run them with the repository virtual environment: `/Users/sai/WS/data-engineering-interview-questions/.venv/bin/python`.

The exercises model the arithmetic that Iceberg procedures use. Compaction output is `ceil(total_bytes / target_file_size)`, the rewrite trigger uses the `delete-ratio-threshold` default of 0.3, and snapshot expiry combines an age rule with `retain_last`. Treat the asserted numbers as a check on the formula, not as a prediction of a real table.

### Try it 1. Choose a partition transform

**Task.** For each query pattern, choose a transform: `days` for a timestamp range filter, `identity` for a low-cardinality equality filter, and `bucket(64, ...)` for a high-cardinality equality filter.

**Hint.** Compare the column cardinality with a threshold of 1000 to separate low from high cardinality, and treat the timestamp column as a range predicate.

**Expected result.** `{'order_ts': 'days(order_ts)', 'customer_id': 'bucket(64, customer_id)', 'region': 'identity(region)'}`.

In [ ]:
patterns = {
    "order_ts": ("range", 90),
    "customer_id": ("equality", 2_000_000),
    "region": ("equality", 40),
}


def choose_transform(column, kind, cardinality):
    if kind == "range":
        return f"days({column})"
    if cardinality <= 1000:
        return f"identity({column})"
    return f"bucket(64, {column})"


transforms = {
    col: choose_transform(col, kind, card)
    for col, (kind, card) in patterns.items()
}
for col, transform in transforms.items():
    print(f"{col:12s} -> {transform}")

assert transforms["order_ts"] == "days(order_ts)"
assert transforms["customer_id"] == "bucket(64, customer_id)"
assert transforms["region"] == "identity(region)"

### Try it 2. Plan a compaction run from a file-size histogram

**Task.** Given a histogram of file sizes and counts, count the files under 128 MB and compute how many target-size files a bin-packing rewrite produces at a 512 MB target.

**Hint.** `small = count where size < 128 MB`; `output_files = ceil(total_bytes / target_bytes)`.

**Expected result.** `5500` small files and `53` output files.

In [ ]:
import math

MB = 1024 ** 2
# size in bytes -> number of files
histogram = {
    1 * 1024: 4000,          # 1 KB
    8 * MB: 1500,            # 8 MB
    300 * MB: 50,            # 300 MB
}
target_bytes = 512 * MB

small_files = sum(count for size, count in histogram.items() if size < 128 * MB)
total_bytes = sum(size * count for size, count in histogram.items())
output_files = math.ceil(total_bytes / target_bytes)

print(f"small files (<128MB) = {small_files}")
print(f"total bytes = {total_bytes / (1024 ** 3):.2f} GiB")
print(f"compaction output files at 512MB target = {output_files}")

assert small_files == 5500
assert output_files == 53

### Try it 3. Compute snapshot expiry retention

**Task.** Given snapshots one day apart, a policy of `older_than` 7 days and `retain_last` 10, compute how many snapshots survive. A snapshot survives if it is within the age window or among the most recent `retain_last`, and the current snapshot always survives.

**Hint.** Build the retained set as the union of `days_ago <= 7` and the first 10 snapshots in newest-first order.

**Expected result.** `10` snapshots survive and `15` are expired.

In [ ]:
# newest first: (snapshot_id, days_ago)
snapshots = [(1000 - i, i) for i in range(25)]
current_id = snapshots[0][0]
older_than_days = 7
retain_last = 10

recent_ids = {sid for sid, _ in snapshots[:retain_last]}
young_ids = {sid for sid, days in snapshots if days <= older_than_days}
retained = recent_ids | young_ids | {current_id}
expired = [sid for sid, _ in snapshots if sid not in retained]

print(f"retained = {len(retained)}")
print(f"expired  = {len(expired)}")

assert len(retained) == 10
assert len(expired) == 15
assert current_id in retained

### Try it 4. Detect small-file growth from manifest counts

**Task.** Given each day's new files and total bytes, compute the average file size and flag days where it drops below half the 256 MB target.

**Hint.** `average_mb = bytes / files`; compare with `0.5 * target_mb`.

**Expected result.** The degraded days are `['2026-10-03', '2026-10-04']`.

In [ ]:
target_mb = 256
daily = {
    "2026-10-01": {"files": 100, "bytes": 100 * target_mb},
    "2026-10-02": {"files": 200, "bytes": 200 * target_mb},
    "2026-10-03": {"files": 900, "bytes": 300 * target_mb},
    "2026-10-04": {"files": 2000, "bytes": 400 * target_mb},
}

degraded = []
for day, stats in daily.items():
    average_mb = stats["bytes"] / stats["files"]
    print(f"{day}: {stats['files']:>5} files, avg {average_mb:6.1f} MB")
    if average_mb < 0.5 * target_mb:
        degraded.append(day)

print("degraded days:", degraded)
assert degraded == ["2026-10-03", "2026-10-04"]

### Try it 5. Estimate planning cost from manifest count

**Task.** Estimate scan-planning time for a table with 40 manifests and for one with 4000, using 64 MB/s metadata read throughput, 2 MB per manifest, and 1.5 ms to parse each manifest. Decide whether a manifest rewrite is warranted when planning exceeds 10 seconds.

**Hint.** `seconds = manifests * size_mb / throughput_mb_s + manifests * parse_ms / 1000`.

**Expected result.** `1.31` seconds for 40 manifests, `131.0` seconds for 4000, and the verdict `rewrite manifests`.

In [ ]:
THROUGHPUT_MB_S = 64
MANIFEST_MB = 2
PARSE_MS = 1.5


def planning_seconds(manifests):
    read_s = manifests * MANIFEST_MB / THROUGHPUT_MB_S
    parse_s = manifests * PARSE_MS / 1000
    return read_s + parse_s


small = planning_seconds(40)
large = planning_seconds(4000)
verdict = "rewrite manifests" if large > 10 else "leave manifests"

print(f"40 manifests   -> {small:.2f} s")
print(f"4000 manifests -> {large:.1f} s")
print("verdict:", verdict)

assert round(small, 2) == 1.31
assert round(large, 1) == 131.0
assert verdict == "rewrite manifests"

### Try it 6. Simulate a snapshot-based rollback

**Task.** Given a parent chain of snapshots and a current snapshot, verify that a target is an ancestor, roll back to it, and show that the snapshots after it are no longer ancestors.

**Hint.** Walk the chain from any snapshot to its parent until you reach `None`.

**Expected result.** Snapshot `3` is an ancestor, the rollback makes it current, and `4` and `5` are no longer ancestors.

In [ ]:
# snapshot id -> parent snapshot id
parents = {1: None, 2: 1, 3: 2, 4: 3, 5: 4}
current = 5


def ancestors(snapshot_id):
    chain = []
    while snapshot_id is not None:
        chain.append(snapshot_id)
        snapshot_id = parents[snapshot_id]
    return chain


target = 3
print("ancestors of current:", ancestors(current))
assert target in ancestors(current)

current = target  # rollback_to_snapshot
new_snapshot = 6
parents[new_snapshot] = current  # the next append builds on the rolled-back state

print("current after rollback:", current)
print("ancestors after rollback:", ancestors(new_snapshot))
assert current == 3
assert 4 not in ancestors(new_snapshot) and 5 not in ancestors(new_snapshot)

### Try it 7. Decide merge-on-read compaction from delete ratios

**Task.** Using the `delete-ratio-threshold` default of 0.3, find the file groups that a `rewrite_data_files` run would rewrite, and total the delete files it would remove.

**Hint.** Rewrite when `delete_ratio >= 0.3`; sum `delete_files` for the selected groups.

**Expected result.** `3` groups rewritten and `170` delete files removed.

In [ ]:
DELETE_RATIO_THRESHOLD = 0.3

groups = [
    {"partition": "p1", "delete_ratio": 0.05, "delete_files": 2},
    {"partition": "p2", "delete_ratio": 0.31, "delete_files": 40},
    {"partition": "p3", "delete_ratio": 0.30, "delete_files": 10},
    {"partition": "p4", "delete_ratio": 0.60, "delete_files": 120},
]

to_rewrite = [g for g in groups if g["delete_ratio"] >= DELETE_RATIO_THRESHOLD]
deletes_removed = sum(g["delete_files"] for g in to_rewrite)

for group in groups:
    flag = "rewrite" if group in to_rewrite else "keep"
    print(f"{group['partition']}: ratio {group['delete_ratio']:.2f} -> {flag}")

print(f"groups rewritten = {len(to_rewrite)}, delete files removed = {deletes_removed}")
assert len(to_rewrite) == 3
assert deletes_removed == 170

## Real world use cases for Apache Iceberg

### 1. Multi-engine lakehouse on object storage

A retailer keeps one copy of curated data in S3 as Iceberg. Spark writes it, Trino serves ad-hoc analytics, Athena powers a serverless dashboard, and Snowflake reads selected tables for finance - all against the same files and the same snapshot.

In [ ]:
# ILLUSTRATIVE - needs the live engine session and data this section names; read on for the pattern.
spark.sql("""
    CREATE TABLE IF NOT EXISTS lakehouse.orders (
        order_id     STRING,
        customer_id  STRING,
        order_ts     TIMESTAMP,
        amount       DECIMAL(18, 2),
        status       STRING
    )
    USING iceberg
    PARTITIONED BY (days(order_ts), bucket(16, customer_id))
""")

(
    spark.read.parquet("s3://acme-raw/orders/ds=2026-10-04/")
    .writeTo("lakehouse.orders")
    .append()
)

Why Iceberg: hidden partitioning means readers do not need to know the partition layout, and partition evolution lets the layout change without rewriting history. A single format removes the copy-per-engine pattern.

### 2. Streaming CDC with upserts and deletes

Flink or Spark Structured Streaming consumes Debezium events and applies them to Iceberg with `MERGE`. Equality deletes let a streaming job mark rows deleted without rewriting whole partitions.

```sql
MERGE INTO lakehouse.customers AS t
USING cdc_batch AS s
ON t.customer_id = s.customer_id
WHEN MATCHED AND s.op = 'd' THEN DELETE
WHEN MATCHED THEN UPDATE SET name = s.name, email = s.email, updated_at = s.updated_at
WHEN NOT MATCHED THEN INSERT (customer_id, name, email, updated_at)
VALUES (s.customer_id, s.name, s.email, s.updated_at);
```

Why Iceberg: row-level deletes and ACID commits make the lakehouse table behave like a mutable database table without nightly full rebuilds.

### 3. GDPR deletion and right-to-be-forgotten

A deletion request must remove one customer's rows from a 500 TB table. Rewriting every partition is unacceptable, so the pipeline deletes by predicate and then expires the old snapshots that still contain the data.

In [ ]:
# ILLUSTRATIVE - needs the live engine session and data this section names; read on for the pattern.
spark.sql("DELETE FROM lakehouse.events WHERE customer_id = 'C-12345'")

# physically remove files that are no longer referenced
spark.sql("CALL prod.system.rewrite_data_files(table => 'lakehouse.events')")
spark.sql("CALL prod.system.expire_snapshots(table => 'lakehouse.events', older_than => TIMESTAMP '2026-10-01 00:00:00')")

Why Iceberg: predicate deletes plus snapshot expiry give an auditable path to true removal, which copy-on-write-parquet pipelines cannot offer without rewriting everything.

### 4. Reproducible reporting and incident recovery

A finance report must be reproducible as of its publication date. Each successful load tags a snapshot; the reporting job pins that snapshot, and a bad deployment is rolled back by pointing the table at the previous snapshot.

In [ ]:
# ILLUSTRATIVE - needs the live engine session and data this section names; read on for the pattern.
spark.sql("ALTER TABLE lakehouse.orders CREATE TAG finance_close_2026_09 AS OF CURRENT_TIMESTAMP")

# read exactly what finance signed off on
finance = spark.read.option("snapshot-id", tag_id).table("lakehouse.orders")

# roll back a bad publish
spark.sql("CALL prod.system.rollback_to_snapshot('lakehouse.orders', 1234567890)")

Why Iceberg: snapshot isolation gives point-in-time reads and fast rollback that do not depend on restoring backups.

### 5. Migration from Hive and Parquet with a zero-downtime cutover

A team migrates partitioned Parquet tables in place, keeping the same storage paths, and dual-runs readers during a validation window.

```sql
-- register existing Parquet files as an Iceberg table without moving data
CALL prod.system.migrate('analytics.orders_parquet');
```

In [ ]:
# ILLUSTRATIVE - needs the live engine session and data this section names; read on for the pattern.
# validate row counts across the old and new access paths before cutover
old = spark.read.parquet("s3://lakehouse/analytics/orders/")
new = spark.table("lakehouse.orders")
assert old.count() == new.count()

Why Iceberg: in-place migration avoids a second copy, and snapshot isolation makes the validation window safe for existing readers.

All five cases rest on the same three mechanics: an atomic pointer swap gives readers a stable snapshot, hidden partitioning plus column stats give pruning without a physical layout column, and delete files plus snapshot expiry give row-level removal with an auditable trail. The cost is maintenance: compaction, manifest rewrites, and snapshot expiry are part of running the table, not optional cleanup.

Common mistake in these programs: running maintenance only when queries get slow. Schedule compaction and snapshot expiry from the start, and watch file counts and manifest counts so you notice small-file growth before it reaches planning.